In [ ]:
import sys
sys.path.append("/users/wx21978/projects/pion-argon-xs-analysis/analysis")

from python.analysis import Master
from apps.cex_analysis_input import BeamPionSelection
from python.analysis.Application import ApplicationArguments
from python.analysis import cross_section as xs

import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import scipy.stats as scistats
from matplotlib.lines import Line2D
import matplotlib.colorizer as colourisers

In [ ]:
# On hm01
# Config has some masks with selections
config_path = "/storage/0/wx21978/phys/2GeV_sample/2GeV_sample_config.json"
config = ApplicationArguments.ResolveConfig(Master.LoadConfiguration(config_path))

# Defines the files
sim_file = Master.FileDescriptor(
    "/storage/0/wx21978/phys/2GeV_ntuples/PDSPProd4a_MC_2GeV_reco1_sce_datadriven_v1_ntuple_v09_41_00_03.root",
    "PDSPAnalyser",
    2)
data_file = Master.FileDescriptor(
    "/storage/0/wx21978/phys/2GeV_ntuples/PDSPProd4_data_2GeV_reco2_ntuple_v09_42_03_01.root",
    "PDSPAnalyser",
    1)

In [ ]:
# Convert to data format
sim_evts_presel = Master.Data(sim_file)
data_evts_presel = Master.Data(data_file)

In [ ]:
reload(cai)

In [ ]:
# Applies beam selection
sim_evts = BeamPionSelection(sim_evts_presel, config, True)
data_evts = cai.BeamPionSelection(data_evts_presel, config, False)

## Format of data
The events objects are slightly difference for simulation (MC) and data samples.
Both contain a `RecoParticleData` object, which contains information about the reconstructed objects.

The simulation additionally contains `TrueParticles` and `TrueParticlesBT`.
These contain information about the simulation truth in the event, and will be discussed more in detail later.

## RecoParticleData
Let's begin with the reconstructed particle data.
This is an object that contains a series of properties, which contain various pieces of information in from the reconstruction.
These properties generally come from PFOs (Particle Flow Objects) which are constructed by the reconstruction that try to group together energy in the detector produced by a single underlying particle (but is not perfect at this!).
The documentation list is supplied in the next cell (but it's not actually complete!).
The code is in the `Master.py` module.

There a 3 types of property.
 - Event/beam particle information. These normally have the `beam_` prefix. The beam particle is one of the PFOs which is treated specially by the reconstruction. If done correctly, this will be the particle that came from the beam from CERN. This is an awkward array with one entry per event.
 - PFO information. These summary properties from a PFO in the detector, such as energy, and position information. Note that the beam PFO is not included in this information. These are awkward arrays with an unknown quantity per event.
 - Hit information. This contains information about the energy and positon of each hit (energy deposit) that makes up a PFO. In principle, the could be obtained for every PFO, however here, only the beam particle has hit-level information. This wouldn't be looked at in this notebook.

 - beam_number (ak.Array): beam PFO number
 - slice_id (ak.Array): slice the PFO corresponds to
 - beam_sliceID (ak.Array): slice the beam PFO corresponds to
 - beam_cosmicScore (ak.Array): whether the reconstruction chain used was for a neutrino vertex (beam) or cosmics
 - beam_endPos (ak.Record): end point of the beam particle
 - beam_startPos (ak.Record): start point of the beam particle
 - beam_caloWire (ak.Array): wire numbers of beam calorimetry
 - beam_michelScore (ak.Array): beam particle michel score
 - beam_nHits (ak.Array): beam particle number of hits
 - beam_dEdx (ak.Array): beam particle dEdx
 - pandora_tag (ak.Array): label given to particles by pandora; track, shower or -999
 - number (ak.Array): PFO number
 - mother (ak.Array): number of mother PFO
 - n_hits (ak.Array): number of hits
 - n_hits_collection (ak.Array): number of collection plane hits
 - energy (ak.Array):
 - momentum (ak.Record):
 - shower_direction (ak.Record):
 - shower_start_pos (ak.Record):
 - shower_length (ak.Array): length of shower (if applicable)
 - showerConeAngle (ak.Array): width of shower (if applicable)
 - em_score (ak.Array): shower like score
 - track_score (ak.Array): track like score
 - cnn_score (ak.Array): em_score/(em_score + track_score)
 - space_points (ak.Record): hit space points
 - channel (ak.Array): hit channel
 - peak_time (ak.Array): hit peak time
 - integral (ak.Array): hit integral
 - hit_energy (ak.Array): hit energy
 - track_dEdX (ak.Array): dEdX (if PFO is interpreted as track like)

### Beam-type information

One of the known issues with the simulation is that the simulated momentum distribution does not match the real moment distribution that we observe.
There are corrections to this in the analysis, but for now, let's just take a look.
This is beam-type information, since there is just one measurement per event, corresponding to the beam particle. But remember there is a failure mode where the PFO selected doesn't match teh particle that went through the beam instrumentation.

Below I fetch the two variables from the simulation and data of measured beam instrumentation momentum.
Try making some plots to compare (but not that the events aren't correlated between simulation and data, so e.g, a 2D hist would not be appropraite).

In [ ]:
sim_beam_momentum = sim_evts.recoParticles.beam_inst_P
print(sim_beam_momentum)
data_beam_momentum = data_evts.recoParticles.beam_inst_P
print(data_beam_momentum)

In [ ]:
# TODO compare these distributions

### PFO type information
A key concept to wrap your head around is the differnce between event-level (beam-type) and PFO-level information.
You must ensure you know, when making a selection, if you are making an event-level or PFO-level selection.
Let's start exploring that here.

First, let's plot some distributions about the PFOs.
The number of hits is the number of energy deposits associated to an energy event.
It's hard to derive too much meaning from comparing these parameters between simulation and data, so for not, we will just look at the simulation.
First, plot the distrubtion of `n_hits` for all PFOs.
Note that, to get awkward arrays to play nicely in a histogram, you will need to flatten the PFO arrays (this must be done only at the very end, since you lose the event indexing):

```ak.ravel(sim_n_hits_pfo)```

In [ ]:
sim_n_hits_pfo = sim_evts.recoParticles.n_hits
print(sim_n_hits_pfo)

# TODO plot the distribution of all PFOs

We can to other thinks with the PFO qunaities.
For example, we might want to know the number of PFOs in an event.
There is no direct variable for this.
But the `n_hits` array we have looked at before has one entry per PFO in each event, so by simply looking at the structure of this array (or any other PFO-level array), we can infer this.
This is left as an exercise to you.

Hint: there are two methods provided by awkward to count entries in arrays.
`ak.num` and `ak.count`.
Read through the documentation from awkard to work out which, and plot this distribution.
Note that now we go back to one value per event, but this value is determined by the PFO-level information.

https://awkward-array.org/doc/main/index.html

In [ ]:
# TODO plot of distribution of the number of PFOs in an event

And we can combine all these types of masks in different ways.
In PDSP, there many PFOs that are caused by cosmic rays (muons) which are unrelated to the beam particle.
Let's try to identify these, and plot how many PFOs are related to the beam particle in an event.

What do we know about the beam vertex?
You should be able to find a `beam_endPos` property.
If the reconstruction is working, this should give us a good idea where the beam interaction occured.

How can we use this information?
Any charged products of the beam which thus begin very close to the beam vertex.
One of the properties we have access too is the `track_start_pos`.
So, let's assume for now that if a particle is a beam daughter it will be near the beam vertex.
For now, we'll go for within 10cm, we'll see how good that is later.
We can then make a mask to select the PFO we think were produced by the beam interaction.

In [ ]:
sim_beam_end = sim_evts.recoParticles.beam_endPos
print(sim_beam_end)
# This is a set of coordinates, can get individuals as follows:
print(sim_beam_end.x, sim_beam_end.y, sim_beam_end.z)

sim_track_start_pfo = sim_evts.recoParticles.track_start_pos # Also coordinate

def get_distance(vertex, track_start):
    # Awkward is nice in that it will "broadcast" the 1 entry per row of the event-level verex
    # to the N entries per row PFO-level information. This is simlar to numpy broadcasting.
    # This means we can naively combine the two, and it will give us a value for each PFO
    # based on the contents of that event only.
    return np.sqrt((vertex.x - track_start.x)**2
                   + (vertex.y - track_start.y)**2
                   + (vertex.z - track_start.z)**2)

distances = get_distance(sim_beam_end, sim_track_start_pfo)
daughter_mask_pfo = distances < 10 # Units are cm

In [ ]:
# TODO use the mask to plot this distribution of the number of selected PFOs per event.
# Hint if you sum a mask, True=1 and False=0.

## TrueParticlesBT
This is Back-Tracked (BT) information.
The simulation particles contain the "physics" truth that was used, i.e. what happened without any detector errors getting in the way.

In the simulation, there is a tool which tracks what particle produced some energy deposit in the detector, then follow that through to the reconstructed hit, and ultimately the PFO.
The back-tracker looks at each of the PFOs in the event and asks which true particle contributed the most to the energy contained within the PFO.
Then, it returns a PFO-like object, but instead of the reconstructed information, it replaces that with the truth information from the particle with the highest energy contribution to that PFO.

Long story short, this gives us an imperfect view of the true properies of whatever created a PFO.
In relation to our previous question, we want to know if a PFO was actually a daughter of the beam?

The truth contains unique identifying `number`s.
The true beam particles in each event are always assigned with 1.
There is a `mother` property which provides not the ID of that particle, but the ID of the particle that produced the current particle through some interaction.
This means, if a PFO has `mother == 1`, then it was produced by an interaction of the beam particle.

 - number (ak.Array): backtracked particle number
 - mother (ak.Array): backtracked particle number of mother
 - pdg (ak.Array): pdg code of backtracked particle
 - motherPdg (ak.Array): pdg code of mother of backtracked particle
 - shower_start_pos (ak.Record):
 - endPos (ak.Record):
 - momentum (ak.Record):
 - energy (ak.Array):
 - energyByHits (ak.Array): energy of the true particle, calculated from the true hits
 - direction (ak.Record):
 - hitsInRecoCluster (ak.Array): total hits in a reco cluster
 - n_hits (ak.Array): true particle hits
 - sharedHits (ak.Array): hits shared by both reco and backtracked true particle
 - trueBeamVertex (ak.Record): end poisition of backtracked true beam particle
 - purity (ak.Array): fraction of shared hits in reconstructed object
 - completeness (ak.Array): fraction of shared hits in the backtrackted true particle
 - hitsInRecoCluster_collection (ak.Array): total hits in a reco cluster
 - n_hits_collection (ak.Array): true particle hits
 - sharedHits_collection (ak.Array): hits shared by both reco and backtracked true particle
 - trueBeamVertex_collection (ak.Record): end poisition of backtracked true beam particle
 - purity_collection (ak.Array): fraction of shared hits in reconstructed object
 - completeness_collection (ak.Array): fraction of shared hits in the backtrackted true particle
 - space_points (ak.Record): hit space points
 - channel (ak.Array): hit channel
 - peak_time (ak.Array): hit peak time
 - integral (ak.Array): hit integral
 - hit_energy (ak.Array): hit energy

In [ ]:
sim_mother_bt_pfo = sim_evts.trueParticlesBT.mother
true_daughter_mask_pfo = sim_mother_bt_pfo == 1

# TODO the BT information always has the same structure as the PFO information, since they
# refer to the same objects. Make some plots to see how well we found the beam daughters!
# Try to construct an "efficiency" - the percentage of the true daughters we found,
# and a "purity" - the percentage of selected PFOs which are actually daughters 
# (i.e. given we selected an event, what is the chance it is a daughter, and not something else random)

In [ ]:
# TODO now we can look at the performance, you can plot the distances
# split by true daughters and other particles. From this, you can asses
# what a sensible cut should be (i.e. instead of 10cm).

## TrueParticleData
One of the other failure modes is missing PFOs from merging between the energy desposits of two particles, or due to neutral particles.
The `TrueParticleData` provides us all the truth information of the event.
To save space, only the information from the beam particle, and associated daughters is actually saved, so you won't find any cosmic rays.
Note that, for a variety of reasons, the number of entries per events is equivalent to those in the `RecoParticleData`, so to compare, one must first contruct event level information for both datasets.

By extracting the information from this truth, we can understand how many particles we are missing due to detector limitations, rather than our own selection strategies.

 - pdg (ak.Array): pdg code
 - number (ak.Array): particle number
 - mother (ak.Array): number of mother particle
 - mother_pdg (ak.Array): mother pdg code
 - mass (ak.Array):
 - energy (ak.Array):
 - momentum (ak.Record):
 - direction (ak.Record):
 - shower_start_pos (ak.Record):
 - endPos (ak.Record):
 - pi0_MC (bool): check if we are looking at pure pi0 MC or beam MC
 - PrimaryPi0Mask (ak.Array): Pi0 produced from beam or generated by particle gun
 - truePhotonMask (ak.Array): get mask of photons which are daughter of the primary pi0

In [ ]:
sim_mother_truth = sim_evts.trueParticles.mother
true_daughter_mask_truth = sim_mother_truth == 1

# TODO make some comparison plots!